In [2]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/workspaces/SebastianBot")

# Wire eventgrid triggered function

In [3]:
from cloud.functions.side_effects.create_calendar_event.models import (
    CreateCalendarEventEventGrid,
)
from cloud.helper.event_grid import EventGridModel


topic_name = "CreateCalendarEvent"
function_name = "create_calendar_event"
subscription_name = topic_name
eventgrid_model = CreateCalendarEventEventGrid

assert issubclass(eventgrid_model, EventGridModel)

In [4]:
from pydantic import BaseModel
from pathlib import Path


class AzureConfig(BaseModel):
    subscription_id: str
    resource_group_other: str
    resource_group_functions: str
    function_app_name: str
    location: str


config = AzureConfig.model_validate_json(Path("azure_config.json").open().read())

In [5]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.eventgrid import EventGridManagementClient

credential = DefaultAzureCredential()

eventgrid_client = EventGridManagementClient(
    credential=credential,
    subscription_id=config.subscription_id,
)

## Create EventGrid topic

In [6]:
from azure.core.exceptions import ResourceNotFoundError
from azure.mgmt.eventgrid.models import Topic


def topic_exists(
    client: EventGridManagementClient, resource_group_name: str, name: str
) -> bool:
    try:
        client.topics.get(
            resource_group_name=resource_group_name,
            topic_name=name,
        )
        return True
    except ResourceNotFoundError:
        return False


def create_topic(
    topic_name: str,
    topic_resource_group: str,
    location: str,
    client: EventGridManagementClient,
):

    if topic_exists(client, topic_resource_group, topic_name):
        print(
            f"Topic '{topic_name}' already exists in resource group '{topic_resource_group}'."
        )
        return

    poller = client.topics.begin_create_or_update(
        resource_group_name=topic_resource_group,
        topic_name=topic_name,
        topic_info=Topic(location=location),
    )
    created_topic = poller.result()
    print(f"Topic '{created_topic.name}' created at {created_topic.endpoint}")

In [7]:
create_topic(topic_name, config.resource_group_other, config.location, eventgrid_client)

Topic 'CreateCalendarEvent' already exists in resource group 'Sebastian'.


## Create subscription to Azure Function

## Check if subscription exists

In [8]:
from azure.mgmt.eventgrid.models import (
    EventSubscription,
    AzureFunctionEventSubscriptionDestination,
)

topic_scope = (
    f"/subscriptions/{config.subscription_id}"
    f"/resourceGroups/{config.resource_group_other}"
    f"/providers/Microsoft.EventGrid/topics/{topic_name}"
)


def subscription_exists(
    scope: str, name: str, client: EventGridManagementClient
) -> bool:
    try:
        client.event_subscriptions.get(
            scope=scope,
            event_subscription_name=name,
        )
        return True
    except ResourceNotFoundError:
        return False


def create_subscription(
    topic_scope: str,
    subscription_name: str,
    config: AzureConfig,
    client: EventGridManagementClient,
):
    if subscription_exists(topic_scope, subscription_name, client):
        print(
            f"Subscription '{subscription_name}' already exists for scope '{topic_scope}'."
        )
        return

    function_resource_id = (
        f"/subscriptions/{config.subscription_id}"
        f"/resourceGroups/{config.resource_group_functions}"
        f"/providers/Microsoft.Web/sites/{config.function_app_name}"
        f"/functions/{function_name}"
    )

    destination = AzureFunctionEventSubscriptionDestination(
        resource_id=function_resource_id,
    )
    poller = eventgrid_client.event_subscriptions.begin_create_or_update(
        scope=topic_scope,
        event_subscription_name=subscription_name,
        event_subscription_info=EventSubscription(destination=destination),
    )
    created_subscription = poller.result()
    print(f"Subscription '{created_subscription.name}' created")

In [9]:
create_subscription(topic_scope, subscription_name, config, eventgrid_client)

Subscription 'CreateCalendarEvent' already exists for scope '/subscriptions/fcdc0c79-2df2-4d19-8f7e-607a4fa79363/resourceGroups/Sebastian/providers/Microsoft.EventGrid/topics/CreateCalendarEvent'.


## Set env vars

### Get topic endpoint and key

In [ ]:
def get_topic_info(resource_group_name: str, topic_name: str) -> tuple[str, str]:
    topic = eventgrid_client.topics.get(
        resource_group_name=resource_group_name,
        topic_name=topic_name,
    )
    endpoint = topic.endpoint
    assert endpoint is not None, "Topic endpoint is None"

    keys = eventgrid_client.topics.list_shared_access_keys(
        resource_group_name=resource_group_name,
        topic_name=topic_name,
    )
    key = keys.key1
    assert key is not None, "Topic key1 is None"

    return endpoint, key

In [11]:
endpoint, key = get_topic_info(config.resource_group_other, topic_name)
print(f"{endpoint=}")
# print(f"{key=}")

endpoint='https://createcalendarevent.germanywestcentral-1.eventgrid.azure.net/api/events'


### Set azure env var

In [12]:
import json
from azure.mgmt.web import WebSiteManagementClient


def update_appsettings(
    config: AzureConfig,
    settings_name: str,
    endpoint: str,
    key: str,
    credential: DefaultAzureCredential,
):
    web_client = WebSiteManagementClient(
        credential=credential,
        subscription_id=config.subscription_id,
    )

    setting_value = json.dumps({"uri": endpoint, "key": key})

    # Fetch existing settings so we don't overwrite them
    existing = web_client.web_apps.list_application_settings(
        resource_group_name=config.resource_group_functions,
        name=config.function_app_name,
    )
    assert existing.properties is not None
    existing.properties[settings_name] = setting_value

    web_client.web_apps.update_application_settings(
        resource_group_name=config.resource_group_functions,
        name=config.function_app_name,
        app_settings=existing,
    )
    print(f"Set '{settings_name}' on {config.function_app_name}")

In [13]:
update_appsettings(config, eventgrid_model.env_name(), endpoint, key, credential)

Set 'CreateCalendarEventEventGrid' on Sebastian


### Set local env var

In [14]:
local_settings_path = Path("/workspaces/SebastianBot/local.settings.json")
local_settings = json.loads(local_settings_path.read_text())

# json.dumps produces the escaped-quote format required by local.settings.json
local_settings["Values"][eventgrid_model.env_name()] = json.dumps(
    {"uri": endpoint, "key": key}
)

local_settings_path.write_text(json.dumps(local_settings, indent=2))
print(f"Updated '{eventgrid_model.env_name()}' in local.settings.json")

Updated 'CreateCalendarEventEventGrid' in local.settings.json
